In [ ]:
import torch
import torch.nn as nn

        # 图像patch数量（序列长度）与Embedding维度是两个完全独立的超参数，它们之间没有直接的数学约束关系
        # Embedding维度必须能被头数整除: 多头自注意力的核心思想是将高维空间的特征表达拆解到多个独立子空间中学习

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads): # 16*16*3=768  16 patches
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == embed_dim, "Embedding维度必须能被头数整除"
        
        '''
            生成QKV的线性变换层
            线性层仅对输入张量的最后一个维度进行变换，其他维度保持不变
        '''
        # output = input @ W.t() + b  # [2,16,768] @ [768,2304] → [2,16,2304]     2304 = 768*3
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)

        self.attn_drop = nn.Dropout(0.3)
        # 再定义一个全连接层，有一个可学习参数Wo，这是为了让他们的输出更好的融合起来
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.proj_drop = nn.Dropout(0.2)

    def forward(self, x):
        B, N, C = x.shape  # 输入形状：(批次大小, 序列长度, 嵌入维度)
        
        # 生成QKV张量, 有的方法是直接通过3个全连接层将Q、K、V分开来计算，这里是直接使用一个全连接层将它们一起计算
        qkv = self.qkv(x) # qkv.shape: torch.Size([2, 16, 2304])   
        
        # 重塑维度：B x N x 3 x self.num_heads x self.head_dim -> 3 x B x num_heads x N x head_dim 
        # 12*64 = 768  768*3 = 2304
        # 
        '''
            这里的3是因为我们要将Q、K、V分开，所以需要将最后一个维度分成3个部分
            'qkv.reshape(B, N, 3, self.num_heads, self.head_dim).shape' # torch.Size([2, 16, 3, 12, 64]) 
            permute(2, 0, 3, 1, 4) # torch.Size([3, 2, 12, 16, 64])
            3是QKV分开，因此把它放在第一维
        '''
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        
        # 分离Q/K/V 
        q, k, v = qkv[0], qkv[1], qkv[2]  # 形状均为(B, num_heads, N, head_dim)
        
        # 缩放点积计算 
        scale = self.head_dim ** -0.5 # 64 ** -0.5 = 0.125
        # @ 符号就是进行矩阵乘法
        print(k.shape) # torch.Size([2, 12, 16, 64])
        print(k.transpose(-2, -1).shape) # torch.Size([2, 12, 64, 16])
        print((q @ k.transpose(-2, -1)).shape) # torch.Size([2, 12, 16, 16])

        attn = (q @ k.transpose(-2, -1)) * scale  # (B, num_heads, N, N)
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        print('attn.shape:',attn.shape) # torch.Size([2, 12, 16, 16])
        print('v.shape:',v.shape) # torch.Size([2, 12, 16, 64])
        print('(attn @ v).shape', (attn @ v).shape) # torch.Size([2, 12, 16, 64])
        print('((attn @ v).transpose(1, 2)).shape', ((attn @ v).transpose(1, 2)).shape) # torch.Size([2, 16, 12, 64])

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        print('x.shape:',x.shape) # torch.Size([2, 16, 768])
        x = self.proj(x)
        x = self.proj_drop(x)
        
        return x

# 设置输入参数
B = 2    # 批次大小
N = 16   # 序列长度（如图像的patch数量）
C = 768  # 嵌入维度
num_heads = 12

# 生成随机输入张量 
x = torch.randn(B, N, C) 

# 实例化多头自注意力模块
mhsa = MultiHeadSelfAttention(embed_dim=C, num_heads=num_heads)

# 前向计算
attn_scores, values = mhsa(x)

print("输入张量形状：", x.shape)      
print("Value张量形状：", values.shape) 
print("注意力分数形状：", attn_scores.shape) 


torch.Size([2, 12, 16, 64])
torch.Size([2, 12, 64, 16])
torch.Size([2, 12, 16, 16])
attn.shape: torch.Size([2, 12, 16, 16])
v.shape: torch.Size([2, 12, 16, 64])
(attn @ v).shape torch.Size([2, 12, 16, 64])
((attn @ v).transpose(1, 2)).shape torch.Size([2, 16, 12, 64])
x.shape: torch.Size([2, 16, 768])
输入张量形状： torch.Size([2, 16, 768])
Value张量形状： torch.Size([16, 768])
注意力分数形状： torch.Size([16, 768])


精简简化版，只是用于查看数据变化

1. 原始张量形状</br>
x 的形状为 [2, 197, 768]，表示：</br>
第 0 维（Batch 维度）: 有 2 个样本。</br>
第 1 维（序列维度）: 每个样本有 197 个元素（如 ViT 中的 [CLS] token + 196 patches）。</br>
第 2 维（特征维度）: 每个元素的特征维度为 768。</br>
2. 切片操作 x[:, 0]</br>
[:, 0] 表示：</br>
: 对第 0 维（Batch 维度）取全部样本（保留 2 个样本）。</br>
0 对第 1 维（序列维度）取索引为 0 的值（即 [CLS] token）。</br>
结果：每个样本的第 1 维被压缩，仅保留第 0 维和第 2 维，形状变为 [2, 768]。</br>

In [39]:
import torch
import torch.nn as nn
from collections import OrderedDict

# -------------------- 1. 定义必要的模块 --------------------
class PatchEmbed(nn.Module):
    """将图像分割为块并嵌入特征"""
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.num_patches = (img_size // patch_size) ** 2

    def forward(self, x):
        x = self.proj(x)           # [B, 768, 14, 14]
        x = x.flatten(2).transpose(1, 2)  # [B, 196, 768]
        return x

# -------------------- 2. 定义ViT模型 --------------------
class VisionTransformer(nn.Module):
    def __init__(self, embed_dim=768, depth=6, num_heads=12, mlp_ratio=4., 
                 representation_size=None, distilled=False):
        super().__init__()
        # 图像分块嵌入层
        self.patch_embed = PatchEmbed()
        # 类别标记和蒸馏标记（此处distilled=False）
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.dist_token = nn.Parameter(torch.zeros(1, 1, embed_dim)) if distilled else None
        # 位置编码
        self.pos_embed = nn.Parameter(torch.zeros(1, self.patch_embed.num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(p=0.1)
        # Transformer编码器（简化为6层）
        self.blocks = nn.Sequential(*[
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, 
                                      dim_feedforward=int(embed_dim*mlp_ratio))
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        # 分类前处理层（按条件初始化）
        if representation_size and not distilled:
            self.pre_logits = nn.Sequential(OrderedDict([
                ("fc", nn.Linear(embed_dim, representation_size)),
                ("act", nn.Tanh())
            ]))
        else:
            self.pre_logits = nn.Identity()
        # 初始化参数
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward_features(self, x):
        # 1. 分块嵌入 [B,3,224,224] → [B,196,768]
        x = self.patch_embed(x)
        # 2. 添加[CLS]标记 [B,196,768] → [B,197,768]
        cls_token = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        # 3. 添加位置编码并Dropout
        x = self.pos_drop(x + self.pos_embed)
        # 4. 通过Transformer编码器 [B,197,768] → [B,197,768]
        x = self.blocks(x)
        # 5. 层归一化
        x = self.norm(x)
        print('x.shape:',x.shape)
        # 6. 提取[CLS]标记并预处理 [B,768]
        x_new = x[:, 0]
        print('x[:, 0].shape:',x_new.shape)
        x_new_1 = x[:, 0, :]
        print('x[:, 0, :].shape:',x_new_1.shape)
        x_new_2 = x[:, 0:3, 1:5]
        print('x[:, 0:3, 1:5].shape:',x_new_2.shape)
        return self.pre_logits(x[:, 0])

# -------------------- 3. 实例化模型并运行 --------------------
if __name__ == "__main__":
    # 参数配置
    embed_dim = 768
    representation_size = None  # 禁用特征投影
    # 实例化模型
    model = VisionTransformer(embed_dim=embed_dim, representation_size=representation_size)
    
    # 生成输入数据: batch_size=2, 3通道, 224x224图像
    x = torch.randn(2, 3, 224, 224)
    print("输入形状:", x.shape)  # [2, 3, 224, 224]
    
    # 前向传播
    features = model.forward_features(x)
    print("输出特征形状:", features.shape)  # [2, 768]


输入形状: torch.Size([2, 3, 224, 224])
x.shape: torch.Size([2, 197, 768])
x[:, 0].shape: torch.Size([2, 768])
x[:, 0, :].shape: torch.Size([2, 768])
x[:, 0:3, 1:5].shape: torch.Size([2, 3, 4])
输出特征形状: torch.Size([2, 768])
